<a href="https://colab.research.google.com/github/Sasidhar015/Data-engineering-training-2026/blob/main/30-10-2026_ASSESMENTS_PySpark/Assesment-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Smart Energy Analytics**

Assesment 1 Solutions

In [1]:
!pip -q install pyspark

from google.colab import files
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window
from pyspark import StorageLevel
import time
import os

uploaded = files.upload()  # Select all six assessment CSV files

spark = (SparkSession.builder
         .appName("EnergyAssessmentSet03")
         .getOrCreate())

def load_csv(filename):
    return spark.read.csv(filename, header=True, inferSchema=True)

readings_df = load_csv("meter_readings_90000.csv")
readings_extra_df = load_csv("meter_readings_extra_12000.csv")
meter_master_df = load_csv("meter_master.csv")
substation_master_df = load_csv("substation_master.csv")
tariff_master_df = load_csv("tariff_master.csv")
maintenance_tickets_df = load_csv("maintenance_tickets_1600.csv")

print("All six CSV files loaded.")

Saving maintenance_tickets_1600.csv to maintenance_tickets_1600.csv
Saving meter_master.csv to meter_master.csv
Saving meter_readings_90000.csv to meter_readings_90000.csv
Saving meter_readings_extra_12000.csv to meter_readings_extra_12000.csv
Saving substation_master.csv to substation_master.csv
Saving tariff_master.csv to tariff_master.csv
All six CSV files loaded.


In [4]:
readings_df.printSchema()
print("Main readings row count:", readings_df.count())
readings_df.show(10, truncate=False)

root
 |-- reading_id: string (nullable = true)
 |-- meter_id: string (nullable = true)
 |-- reading_timestamp: timestamp (nullable = true)
 |-- energy_kwh: double (nullable = true)
 |-- voltage_v: double (nullable = true)
 |-- power_factor: double (nullable = true)
 |-- outage_minutes: integer (nullable = true)
 |-- reading_status: string (nullable = true)
 |-- source_type: string (nullable = true)
 |-- ambient_temperature_c: double (nullable = true)
 |-- operator_id: string (nullable = true)

Main readings row count: 90000
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|reading_timestamp  |energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|RD0000001 |M0

In [6]:
selected_readings_df = readings_df.select(
    "reading_id", "meter_id", "reading_timestamp",
    "energy_kwh", "reading_status"
)

collection_mode_df = readings_df.withColumnRenamed(
    "source_type", "collection_mode"
)

selected_readings_df.show(5, truncate=False)
collection_mode_df.select("reading_id", "collection_mode").show(5, truncate=False)

+----------+--------+-------------------+----------+--------------+
|reading_id|meter_id|reading_timestamp  |energy_kwh|reading_status|
+----------+--------+-------------------+----------+--------------+
|RD0000001 |M0082   |2026-04-01 02:00:00|22.53     |Valid         |
|RD0000002 |M0020   |2026-01-31 06:00:00|2.97      |Valid         |
|RD0000003 |M0054   |2026-04-11 10:15:00|8.61      |Estimated     |
|RD0000004 |M0111   |2026-03-14 00:00:00|1.87      |Valid         |
|RD0000005 |M0135   |2026-05-27 01:45:00|4.39      |Valid         |
+----------+--------+-------------------+----------+--------------+
only showing top 5 rows
+----------+---------------+
|reading_id|collection_mode|
+----------+---------------+
|RD0000001 |SmartMeter     |
|RD0000002 |SmartMeter     |
|RD0000003 |SmartMeter     |
|RD0000004 |Manual         |
|RD0000005 |SmartMeter     |
+----------+---------------+
only showing top 5 rows


In [21]:
high_valid_energy_df = (
    readings_df
    .filter((F.col("reading_status") == "Valid") & (F.col("energy_kwh") > 20))
    .orderBy(F.col("energy_kwh").desc()))

high_valid_energy_df.show(20, truncate=False)

+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|reading_id|meter_id|reading_timestamp  |energy_kwh|voltage_v|power_factor|outage_minutes|reading_status|source_type|ambient_temperature_c|operator_id|
+----------+--------+-------------------+----------+---------+------------+--------------+--------------+-----------+---------------------+-----------+
|RD0041703 |M0101   |2026-03-06 22:30:00|110.0     |213.1    |0.822       |0             |Valid         |SmartMeter |31.1                 |OP010      |
|RD0077494 |M0014   |2026-05-05 17:30:00|109.99    |NULL     |0.771       |0             |Valid         |Manual     |39.0                 |OP031      |
|RD0034412 |M0026   |2026-02-24 21:15:00|109.96    |231.6    |0.964       |0             |Valid         |SmartMeter |25.3                 |OP017      |
|RD0039063 |M0101   |2026-02-01 00:45:00|109.96    |231.6    |0.835       |0            

In [22]:
energy_band_df = readings_df.withColumn("energy_band",
    F.when(F.col("energy_kwh").isNull(), "Unknown")
     .when(F.col("energy_kwh") < 5, "Low")
     .when(F.col("energy_kwh") <= 25, "Medium")
     .otherwise("High"))

energy_band_df.select("reading_id", "energy_kwh", "energy_band").show(15, truncate=False)

+----------+----------+-----------+
|reading_id|energy_kwh|energy_band|
+----------+----------+-----------+
|RD0000001 |22.53     |Medium     |
|RD0000002 |2.97      |Low        |
|RD0000003 |8.61      |Medium     |
|RD0000004 |1.87      |Low        |
|RD0000005 |4.39      |Low        |
|RD0000006 |0.36      |Low        |
|RD0000007 |79.47     |High       |
|RD0000008 |28.35     |High       |
|RD0000009 |1.96      |Low        |
|RD0000010 |22.52     |Medium     |
|RD0000011 |10.51     |Medium     |
|RD0000012 |1.49      |Low        |
|RD0000013 |15.75     |Medium     |
|RD0000014 |15.96     |Medium     |
|RD0000015 |6.06      |Medium     |
+----------+----------+-----------+
only showing top 15 rows


In [23]:
null_counts_df = readings_df.select([
    F.sum(F.col(c).isNull().cast("long")).alias(c + "_null_count")
    for c in ["energy_kwh", "voltage_v", "power_factor", "operator_id"]])

null_counts_df.show(truncate=False)

+---------------------+--------------------+-----------------------+----------------------+
|energy_kwh_null_count|voltage_v_null_count|power_factor_null_count|operator_id_null_count|
+---------------------+--------------------+-----------------------+----------------------+
|1620                 |2070                |1440                   |5400                  |
+---------------------+--------------------+-----------------------+----------------------+



In [24]:
readings_filled_df = readings_df.fillna({
    "operator_id": "UNASSIGNED",
    "power_factor": 0.90})

print("Original DataFrame remains unchanged.")
readings_filled_df.select("reading_id", "operator_id", "power_factor").show(10, truncate=False)

Original DataFrame remains unchanged.
+----------+-----------+------------+
|reading_id|operator_id|power_factor|
+----------+-----------+------------+
|RD0000001 |OP029      |0.939       |
|RD0000002 |OP025      |0.845       |
|RD0000003 |OP007      |0.879       |
|RD0000004 |UNASSIGNED |0.831       |
|RD0000005 |OP019      |0.966       |
|RD0000006 |OP021      |0.788       |
|RD0000007 |OP016      |0.986       |
|RD0000008 |OP025      |0.818       |
|RD0000009 |OP010      |0.817       |
|RD0000010 |OP028      |0.804       |
+----------+-----------+------------+
only showing top 10 rows


In [25]:
readings_clean_df = readings_df.filter(
    F.col("energy_kwh").isNotNull() & F.col("voltage_v").isNotNull())

original_count = readings_df.count()
clean_count = readings_clean_df.count()
removed_count = original_count - clean_count

print("Original rows:", original_count)
print("Rows after removing null energy_kwh / voltage_v:", clean_count)
print("Rows removed:", removed_count)

Original rows: 90000
Rows after removing null energy_kwh / voltage_v: 86349
Rows removed: 3651


In [26]:
readings_meter_df = readings_df.join(
    meter_master_df,
    on="meter_id",
    how="inner")

valid_energy_by_site_df = (
    readings_meter_df
    .filter(F.col("reading_status") == "Valid")
    .groupBy("site_type")
    .agg(F.round(F.sum("energy_kwh"), 3).alias("total_valid_energy_kwh"))
    .orderBy(F.col("total_valid_energy_kwh").desc()))

valid_energy_by_site_df.show(truncate=False)

+---------------+----------------------+
|site_type      |total_valid_energy_kwh|
+---------------+----------------------+
|Industrial     |710317.41             |
|Commercial     |451525.18             |
|Public Facility|138870.41             |
|Residential    |100863.29             |
+---------------+----------------------+



In [28]:
readings_meter_substation_df = readings_meter_df.join(
    substation_master_df,
    on="substation_id",
    how="left")

valid_energy_by_region_city_df = (
    readings_meter_substation_df
    .filter(F.col("reading_status") == "Valid")
    .groupBy("region", "city")
    .agg(F.round(F.sum("energy_kwh"), 3).alias("total_valid_energy_kwh"))
    .orderBy(F.col("total_valid_energy_kwh").desc()))

valid_energy_by_region_city_df.show(truncate=False)

+-------+-------------+----------------------+
|region |city         |total_valid_energy_kwh|
+-------+-------------+----------------------+
|South  |Visakhapatnam|205015.83             |
|South  |Kochi        |204272.46             |
|Central|Indore       |176503.91             |
|West   |Pune         |162777.24             |
|North  |Lucknow      |142610.4              |
|West   |Ahmedabad    |138627.74             |
|North  |Delhi        |94234.89              |
|South  |Bengaluru    |73787.6               |
|West   |Mumbai       |52116.21              |
|South  |Hyderabad    |51937.79              |
|South  |Chennai      |51829.06              |
|North  |Jaipur       |47863.16              |
+-------+-------------+----------------------+



In [29]:
readings_with_tariff_df = (
    readings_meter_df
    .join(tariff_master_df, on="tariff_id", how="left")
    .filter(
        (F.col("reading_status") != "Rejected") &
        F.col("energy_kwh").isNotNull() )
    .withColumn(
        "energy_charge_inr",
        F.round(F.col("energy_kwh") * F.col("rate_per_kwh_inr"), 2)))

readings_with_tariff_df.select(
    "reading_id", "meter_id", "reading_status", "energy_kwh",
    "tariff_id", "rate_per_kwh_inr", "energy_charge_inr").show(10, truncate=False)


+----------+--------+--------------+----------+-------------------+----------------+-----------------+
|reading_id|meter_id|reading_status|energy_kwh|tariff_id          |rate_per_kwh_inr|energy_charge_inr|
+----------+--------+--------------+----------+-------------------+----------------+-----------------+
|RD0000001 |M0082   |Valid         |22.53     |2026-10-30 05:00:00|6.7             |150.95           |
|RD0000002 |M0020   |Valid         |2.97      |2026-10-30 02:00:00|6.4             |19.01            |
|RD0000003 |M0054   |Estimated     |8.61      |2026-10-30 03:00:00|8.2             |70.6             |
|RD0000004 |M0111   |Valid         |1.87      |2026-10-30 01:00:00|5.1             |9.54             |
|RD0000005 |M0135   |Valid         |4.39      |2026-10-30 03:00:00|8.2             |36.0             |
|RD0000006 |M0139   |Valid         |0.36      |2026-10-30 02:00:00|6.4             |2.3              |
|RD0000007 |M0142   |Valid         |79.47     |2026-10-30 04:00:00|9.1   

In [31]:
tickets_with_meter_df = maintenance_tickets_df.join(
    meter_master_df.select("meter_id", "site_type"),
    on="meter_id",
    how="left")

maintenance_summary_df = (
    tickets_with_meter_df
    .groupBy("site_type", "severity")
    .agg(
        F.count("ticket_id").alias("ticket_count"),
        F.round(F.avg("maintenance_cost_inr"), 2).alias("avg_maintenance_cost_inr"))
    .orderBy("site_type", "severity"))

maintenance_summary_df.show(truncate=False)

+---------------+--------+------------+------------------------+
|site_type      |severity|ticket_count|avg_maintenance_cost_inr|
+---------------+--------+------------+------------------------+
|Commercial     |Critical|37          |13424.73                |
|Commercial     |High    |130         |12293.07                |
|Commercial     |Low     |166         |12223.52                |
|Commercial     |Medium  |171         |13706.6                 |
|Industrial     |Critical|17          |12296.2                 |
|Industrial     |High    |63          |12838.87                |
|Industrial     |Low     |66          |12717.53                |
|Industrial     |Medium  |106         |12043.9                 |
|Public Facility|Critical|15          |8986.84                 |
|Public Facility|High    |48          |12905.28                |
|Public Facility|Low     |60          |11195.93                |
|Public Facility|Medium  |97          |13050.63                |
|Residential    |Critical

In [33]:
substation_reliability_df = (
    readings_meter_df
    .groupBy("substation_id")
    .agg(
        F.round(F.avg("voltage_v"), 2).alias("avg_voltage_v"),
        F.sum(F.coalesce(F.col("outage_minutes"), F.lit(0))).alias("total_outage_minutes"))
    .orderBy(F.col("total_outage_minutes").desc()))

substation_reliability_df.show(5, truncate=False)

+-------------+-------------+--------------------+
|substation_id|avg_voltage_v|total_outage_minutes|
+-------------+-------------+--------------------+
|SUB10        |230.04       |91224               |
|SUB09        |230.05       |81734               |
|SUB04        |230.06       |75872               |
|SUB08        |229.88       |75581               |
|SUB03        |230.03       |61662               |
+-------------+-------------+--------------------+
only showing top 5 rows


In [35]:
status_counts_by_source_df = (
    readings_df
    .filter(F.col("reading_status").isin("Valid", "Estimated", "Rejected"))
    .groupBy("source_type", "reading_status")
    .agg(F.count("*").alias("reading_count"))
    .orderBy(F.col("reading_count").desc()))

status_counts_by_source_df.show(truncate=False)

+-----------+--------------+-------------+
|source_type|reading_status|reading_count|
+-----------+--------------+-------------+
|SmartMeter |Valid         |65521        |
|SmartMeter |Estimated     |10443        |
|Manual     |Valid         |8965         |
|SmartMeter |Rejected      |3202         |
|Manual     |Estimated     |1405         |
|Manual     |Rejected      |464          |
+-----------+--------------+-------------+



In [36]:
readings_timestamp_df = readings_meter_substation_df.withColumn(
    "reading_timestamp",
    F.to_timestamp("reading_timestamp"))

monthly_valid_energy_region_df = (
    readings_timestamp_df
    .filter(F.col("reading_status") == "Valid")
    .withColumn("reading_year", F.year("reading_timestamp"))
    .withColumn("reading_month", F.month("reading_timestamp"))
    .groupBy("region", "reading_year", "reading_month")
    .agg(F.round(F.sum("energy_kwh"), 3).alias("monthly_valid_energy_kwh"))
    .orderBy("region", "reading_year", "reading_month"))

monthly_valid_energy_region_df.show(30, truncate=False)

+-------+------------+-------------+------------------------+
|region |reading_year|reading_month|monthly_valid_energy_kwh|
+-------+------------+-------------+------------------------+
|Central|2026        |1            |31475.58                |
|Central|2026        |2            |28161.41                |
|Central|2026        |3            |29805.85                |
|Central|2026        |4            |29475.34                |
|Central|2026        |5            |30048.88                |
|Central|2026        |6            |27536.85                |
|North  |2026        |1            |48849.31                |
|North  |2026        |2            |45444.25                |
|North  |2026        |3            |51094.61                |
|North  |2026        |4            |43451.02                |
|North  |2026        |5            |48796.93                |
|North  |2026        |6            |47072.33                |
|South  |2026        |1            |97547.39                |
|South  

In [38]:
valid_energy_by_meter_sub_df = (
    readings_meter_df
    .filter(F.col("reading_status") == "Valid")
    .groupBy("substation_id", "meter_id")
    .agg(F.sum("energy_kwh").alias("total_valid_energy_kwh")))

meter_rank_window = Window.partitionBy("substation_id").orderBy(
    F.col("total_valid_energy_kwh").desc())

top_three_meters_df = (
    valid_energy_by_meter_sub_df
    .withColumn("energy_rank", F.dense_rank().over(meter_rank_window))
    .filter(F.col("energy_rank") <= 3)
    .orderBy("substation_id", "energy_rank", "meter_id"))

top_three_meters_df.show(50, truncate=False)

+-------------+--------+----------------------+-----------+
|substation_id|meter_id|total_valid_energy_kwh|energy_rank|
+-------------+--------+----------------------+-----------+
|SUB01        |M0130   |9364.98               |1          |
|SUB01        |M0110   |9259.750000000002     |2          |
|SUB01        |M0068   |8832.580000000005     |3          |
|SUB02        |M0071   |29242.04              |1          |
|SUB02        |M0145   |28641.319999999992    |2          |
|SUB02        |M0133   |7060.140000000003     |3          |
|SUB03        |M0096   |9632.740000000003     |1          |
|SUB03        |M0080   |9525.940000000002     |2          |
|SUB03        |M0073   |9369.789999999995     |3          |
|SUB04        |M0038   |29505.88              |1          |
|SUB04        |M0099   |29216.319999999996    |2          |
|SUB04        |M0098   |28613.460000000014    |3          |
|SUB05        |M0094   |9659.639999999996     |1          |
|SUB05        |M0050   |9570.85999999999

In [41]:
valid_readings_for_window_df = readings_df.filter(
    F.col("reading_status") == "Valid").withColumn(
    "reading_timestamp", F.to_timestamp("reading_timestamp"))

cumulative_window = (
    Window.partitionBy("meter_id")
    .orderBy("reading_timestamp", "reading_id")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow))

cumulative_valid_energy_df = valid_readings_for_window_df.withColumn(
    "cumulative_valid_energy_kwh",
    F.sum("energy_kwh").over(cumulative_window))

cumulative_valid_energy_df.select(
    "meter_id", "reading_timestamp", "reading_id",
    "energy_kwh", "cumulative_valid_energy_kwh"
).orderBy("meter_id", "reading_timestamp", "reading_id").show(20, truncate=False)

+--------+-------------------+----------+----------+---------------------------+
|meter_id|reading_timestamp  |reading_id|energy_kwh|cumulative_valid_energy_kwh|
+--------+-------------------+----------+----------+---------------------------+
|M0001   |2026-01-01 01:15:00|RD0083587 |22.27     |22.27                      |
|M0001   |2026-01-01 03:30:00|RD0086275 |4.05      |26.32                      |
|M0001   |2026-01-01 09:45:00|RD0088204 |11.28     |37.6                       |
|M0001   |2026-01-02 04:45:00|RD0005149 |11.76     |49.36                      |
|M0001   |2026-01-02 08:15:00|RD0035530 |10.06     |59.42                      |
|M0001   |2026-01-02 18:30:00|RD0070104 |15.83     |75.25                      |
|M0001   |2026-01-03 00:30:00|RD0006677 |17.06     |92.31                      |
|M0001   |2026-01-03 06:00:00|RD0021042 |17.14     |109.45                     |
|M0001   |2026-01-03 08:30:00|RD0032205 |20.46     |129.91                     |
|M0001   |2026-01-03 18:15:0

In [44]:
main_reading_count = readings_df.count()
extra_reading_count = readings_extra_df.count()

combined_readings_raw_df = readings_df.unionByName(readings_extra_df)
combined_readings_count = combined_readings_raw_df.count()

print("Main readings rows:", main_reading_count)
print("Extra readings rows:", extra_reading_count)
print("Expected combined rows:", main_reading_count + extra_reading_count)
print("Actual union rows:", combined_readings_count)

Main readings rows: 90000
Extra readings rows: 12000
Expected combined rows: 102000
Actual union rows: 102000


In [46]:
duplicate_reading_ids_df = (
    combined_readings_raw_df
    .groupBy("reading_id")
    .count()
    .filter(F.col("count") > 1)
    .orderBy(F.col("count").desc(), "reading_id"))

print("Number of reading_id values repeated:", duplicate_reading_ids_df.count())
duplicate_reading_ids_df.show(20, truncate=False)
combined_readings_dedup_df = combined_readings_raw_df.dropDuplicates(["reading_id"])

print("Rows before deduplication:", combined_readings_count)
print("Unique readings after deduplication:", combined_readings_dedup_df.count())

Number of reading_id values repeated: 700
+----------+-----+
|reading_id|count|
+----------+-----+
|RD0000004 |2    |
|RD0000121 |2    |
|RD0000194 |2    |
|RD0000545 |2    |
|RD0000650 |2    |
|RD0000715 |2    |
|RD0000765 |2    |
|RD0001057 |2    |
|RD0001183 |2    |
|RD0001235 |2    |
|RD0001797 |2    |
|RD0001931 |2    |
|RD0001933 |2    |
|RD0002277 |2    |
|RD0002387 |2    |
|RD0002407 |2    |
|RD0002679 |2    |
|RD0002711 |2    |
|RD0002757 |2    |
|RD0002925 |2    |
+----------+-----+
only showing top 20 rows
Rows before deduplication: 102000
Unique readings after deduplication: 101300


In [48]:
lazy_energy_pipeline_df = (
    readings_df
    .filter((F.col("reading_status") == "Valid") & F.col("energy_kwh").isNotNull())
    .withColumn("energy_charge_proxy", F.col("energy_kwh") * F.lit(1.0))
    .groupBy("meter_id")
    .agg(F.round(F.sum("energy_charge_proxy"), 3).alias("total_energy_proxy"))
    .orderBy(F.col("total_energy_proxy").desc()))

lazy_energy_pipeline_df.explain("formatted")

lazy_energy_pipeline_df.show(10, truncate=False)

== Physical Plan ==
AdaptiveSparkPlan (9)
+- Sort (8)
   +- Exchange (7)
      +- HashAggregate (6)
         +- Exchange (5)
            +- HashAggregate (4)
               +- Project (3)
                  +- Filter (2)
                     +- Scan csv  (1)


(1) Scan csv 
Output [3]: [meter_id#18, energy_kwh#20, reading_status#24]
Batched: false
Location: InMemoryFileIndex [file:/content/meter_readings_90000.csv]
PushedFilters: [IsNotNull(reading_status), EqualTo(reading_status,Valid), IsNotNull(energy_kwh)]
ReadSchema: struct<meter_id:string,energy_kwh:double,reading_status:string>

(2) Filter
Input [3]: [meter_id#18, energy_kwh#20, reading_status#24]
Condition : ((isnotnull(reading_status#24) AND (reading_status#24 = Valid)) AND isnotnull(energy_kwh#20))

(3) Project
Output [2]: [meter_id#18, (energy_kwh#20 * 1.0) AS energy_charge_proxy#1958]
Input [3]: [meter_id#18, energy_kwh#20, reading_status#24]

(4) HashAggregate
Input [2]: [meter_id#18, energy_charge_proxy#1958]
Keys [1]: [me

In [49]:
valid_readings_cached_df = readings_df.filter(
    F.col("reading_status") == "Valid").cache()

start_time = time.time()
valid_count_first = valid_readings_cached_df.count()
first_count_seconds = time.time() - start_time

start_time = time.time()
valid_count_second = valid_readings_cached_df.count()
second_count_seconds = time.time() - start_time

print("Valid count (first action):", valid_count_first)
print("First count elapsed seconds:", round(first_count_seconds, 4))
print("Valid count (second action):", valid_count_second)
print("Second count elapsed seconds:", round(second_count_seconds, 4))
print("DataFrame is cached:", valid_readings_cached_df.is_cached)

valid_readings_cached_df.unpersist()

Valid count (first action): 74486
First count elapsed seconds: 2.6868
Valid count (second action): 74486
Second count elapsed seconds: 0.1211
DataFrame is cached: True


DataFrame[reading_id: string, meter_id: string, reading_timestamp: timestamp, energy_kwh: double, voltage_v: double, power_factor: double, outage_minutes: int, reading_status: string, source_type: string, ambient_temperature_c: double, operator_id: string]

In [59]:
outage_readings_persisted_df = (
    readings_df
    .filter(F.coalesce(F.col("outage_minutes"), F.lit(0)) > 0)
    .persist(StorageLevel.MEMORY_AND_DISK))

outage_rows_count = outage_readings_persisted_df.count()
outage_minutes_total = outage_readings_persisted_df.agg(
    F.sum("outage_minutes").alias("total_outage_minutes")).first()["total_outage_minutes"]

print("Outage reading rows:", outage_rows_count)
print("Total outage minutes:", outage_minutes_total)

outage_readings_persisted_df.unpersist()
print("Persisted outage DataFrame released.")

Outage reading rows: 7653
Total outage minutes: 707706
Persisted outage DataFrame released.


In [60]:
def timed_aggregation(dataframe, label):
    start = time.time()
    result_count = (
        dataframe
        .groupBy("meter_id")
        .agg(F.round(F.sum("energy_kwh"), 3).alias("total_energy_kwh"))
        .count())
    elapsed = time.time() - start
    print(label, "| result groups:", result_count, "| seconds:", round(elapsed, 4))
    return elapsed

without_cache_first = timed_aggregation(readings_df, "Without cache - run 1")
without_cache_second = timed_aggregation(readings_df, "Without cache - run 2")

cached_source_df = readings_df.cache()
cached_source_df.count()
with_cache_first = timed_aggregation(cached_source_df, "With cache - run 1")
with_cache_second = timed_aggregation(cached_source_df, "With cache - run 2")
cached_source_df.unpersist()

print("Observation: compare the measured times above. Caching may help repeated work,")
print("but results depend on cluster resources, data size, cache materialization and runtime overhead.")

Without cache - run 1 | result groups: 150 | seconds: 1.0525
Without cache - run 2 | result groups: 150 | seconds: 0.9959
With cache - run 1 | result groups: 150 | seconds: 0.3126
With cache - run 2 | result groups: 150 | seconds: 0.1933
Observation: compare the measured times above. Caching may help repeated work,
but results depend on cluster resources, data size, cache materialization and runtime overhead.


In [61]:
readings_8_partitions_df = readings_df.repartition(8)

rows_per_partition_df = (
    readings_8_partitions_df
    .withColumn("partition_id", F.spark_partition_id())
    .groupBy("partition_id")
    .agg(F.count("*").alias("row_count"))
    .orderBy("partition_id"))

print("Partition count:", readings_8_partitions_df.rdd.getNumPartitions())
rows_per_partition_df.show(20, truncate=False)

Partition count: 8
+------------+---------+
|partition_id|row_count|
+------------+---------+
|0           |11249    |
|1           |11249    |
|2           |11251    |
|3           |11251    |
|4           |11250    |
|5           |11250    |
|6           |11250    |
|7           |11250    |
+------------+---------+



In [62]:
readings_12_partitions_df = readings_df.repartition(12)
coalesced_3_df = readings_12_partitions_df.coalesce(3)
repartitioned_3_df = readings_12_partitions_df.repartition(3)

print("Starting partitions:", readings_12_partitions_df.rdd.getNumPartitions())
print("After coalesce(3):", coalesced_3_df.rdd.getNumPartitions())
print("After repartition(3):", repartitioned_3_df.rdd.getNumPartitions())

print("\nCOALESCE EXECUTION PLAN")
coalesced_3_df.explain("formatted")

print("\nREPARTITION EXECUTION PLAN")
repartitioned_3_df.explain("formatted")

# coalesce usually reduces partitions with less data movement; repartition performs a shuffle.

Starting partitions: 12
After coalesce(3): 3
After repartition(3): 3

COALESCE EXECUTION PLAN
== Physical Plan ==
AdaptiveSparkPlan (7)
+- == Final Plan ==
   ResultQueryStage (5)
   +- Coalesce (4)
      +- ShuffleQueryStage (3), Statistics(sizeInBytes=13.0 MiB, rowCount=9.00E+4)
         +- Exchange (2)
            +- Scan csv  (1)
+- == Initial Plan ==
   Coalesce (6)
   +- Exchange (2)
      +- Scan csv  (1)


(1) Scan csv 
Output [11]: [reading_id#17, meter_id#18, reading_timestamp#19, energy_kwh#20, voltage_v#21, power_factor#22, outage_minutes#23, reading_status#24, source_type#25, ambient_temperature_c#26, operator_id#27]
Batched: false
Location: InMemoryFileIndex [file:/content/meter_readings_90000.csv]
ReadSchema: struct<reading_id:string,meter_id:string,reading_timestamp:timestamp,energy_kwh:double,voltage_v:double,power_factor:double,outage_minutes:int,reading_status:string,source_type:string,ambient_temperature_c:double,operator_id:string>

(2) Exchange
Input [11]: [readin

In [63]:
cleaned_enriched_df = (
    readings_clean_df
    .join(meter_master_df, on="meter_id", how="inner")
    .join(substation_master_df, on="substation_id", how="left"))

output_path_task25 = "/content/energy_enriched_parquet"

(   cleaned_enriched_df
    .repartition(4)
    .write
    .mode("overwrite")
    .parquet(output_path_task25))

print("Parquet output written to:", output_path_task25)
part_files_task25 = [
    f for f in os.listdir(output_path_task25)
    if f.startswith("part-") and f.endswith(".parquet")]
print("Parquet part files:", part_files_task25)
print("Number of Parquet part files:", len(part_files_task25))

Parquet output written to: /content/energy_enriched_parquet
Parquet part files: ['part-00002-d6f7210c-30d4-48d7-8f70-496150c5d73b-c000.snappy.parquet', 'part-00001-d6f7210c-30d4-48d7-8f70-496150c5d73b-c000.snappy.parquet', 'part-00003-d6f7210c-30d4-48d7-8f70-496150c5d73b-c000.snappy.parquet', 'part-00000-d6f7210c-30d4-48d7-8f70-496150c5d73b-c000.snappy.parquet']
Number of Parquet part files: 4


**Final Capstone**

Energy Consumption & Reliability Report

In [65]:
capstone_union_df = readings_df.unionByName(readings_extra_df)

capstone_duplicate_ids_df = (
    capstone_union_df.groupBy("reading_id")
    .count()
    .filter(F.col("count") > 1))

capstone_readings_df = capstone_union_df.dropDuplicates(["reading_id"])

print("Capstone union row count:", capstone_union_df.count())
print("Repeated reading_id count:", capstone_duplicate_ids_df.count())
print("Deduplicated reading count:", capstone_readings_df.count())


Capstone union row count: 102000
Repeated reading_id count: 700
Deduplicated reading count: 101300


In [66]:
capstone_enriched_df = (
    capstone_readings_df
    .withColumn("reading_timestamp", F.to_timestamp("reading_timestamp"))
    .join(meter_master_df, on="meter_id", how="inner")
    .join(substation_master_df, on="substation_id", how="left")
    .join(tariff_master_df, on="tariff_id", how="left"))

In [68]:
capstone_enriched_df = (
    capstone_enriched_df
    .withColumn("month", F.date_format("reading_timestamp", "yyyy-MM"))
    .withColumn(
        "billable_energy_kwh",
        F.when(
            (F.col("reading_status") != "Rejected") &
            F.col("energy_kwh").isNotNull(),
            F.col("energy_kwh")
        ).otherwise(F.lit(0.0)))
    .withColumn(
        "energy_charge_inr",
        F.when(
            (F.col("reading_status") != "Rejected") &
            F.col("energy_kwh").isNotNull() &
            F.col("rate_per_kwh_inr").isNotNull(),
            F.col("energy_kwh") * F.col("rate_per_kwh_inr")
        ).otherwise(F.lit(0.0)))
    .withColumn(
        "outage_minutes_clean",
        F.coalesce(F.col("outage_minutes"), F.lit(0)))
    .withColumn(
        "outage_flag",
        F.when(F.col("outage_minutes_clean") > 0, 1).otherwise(0)))

capstone_enriched_df.select(
    "reading_id", "month", "region", "site_type", "reading_status",
    "energy_kwh", "billable_energy_kwh", "rate_per_kwh_inr",
    "energy_charge_inr", "outage_minutes_clean", "outage_flag").show(10, truncate=False)

+----------+-------+-------+---------------+--------------+----------+-------------------+----------------+------------------+--------------------+-----------+
|reading_id|month  |region |site_type      |reading_status|energy_kwh|billable_energy_kwh|rate_per_kwh_inr|energy_charge_inr |outage_minutes_clean|outage_flag|
+----------+-------+-------+---------------+--------------+----------+-------------------+----------------+------------------+--------------------+-----------+
|RD0000002 |2026-01|West   |Residential    |Valid         |2.97      |2.97               |6.4             |19.008000000000003|0                   |0          |
|RD0000003 |2026-04|Central|Commercial     |Estimated     |8.61      |8.61               |8.2             |70.60199999999999 |0                   |0          |
|RD0000004 |2026-03|Central|Residential    |Valid         |1.87      |1.87               |5.1             |9.536999999999999 |0                   |0          |
|RD0000005 |2026-05|Central|Commercial  

In [69]:
monthly_energy_reliability_df = (
    capstone_enriched_df
    .groupBy("month", "region", "site_type")
    .agg(
        F.round(F.sum("billable_energy_kwh"), 3).alias("total_billable_kwh"),
        F.round(F.sum("energy_charge_inr"), 2).alias("total_billable_energy_charges_inr"),
        F.sum("outage_minutes_clean").alias("total_outage_minutes"),
        F.count("*").alias("total_readings"),
        F.sum("outage_flag").alias("readings_with_outage"))
    .withColumn(
        "outage_reading_percentage",
        F.round(
            F.when(F.col("total_readings") > 0,
                   F.col("readings_with_outage") * 100.0 / F.col("total_readings"))
             .otherwise(F.lit(0.0)),2)))


In [70]:
rank_window = Window.partitionBy("month", "region").orderBy(
    F.col("total_billable_energy_charges_inr").desc())

In [72]:
monthly_energy_reliability_ranked_df = (
    monthly_energy_reliability_df
    .withColumn("site_type_rank", F.dense_rank().over(rank_window))
    .orderBy("month", "region", "site_type_rank"))

monthly_energy_reliability_ranked_df.show(100, truncate=False)

+-------+-------+---------------+------------------+---------------------------------+--------------------+--------------+--------------------+-------------------------+--------------+
|month  |region |site_type      |total_billable_kwh|total_billable_energy_charges_inr|total_outage_minutes|total_readings|readings_with_outage|outage_reading_percentage|site_type_rank|
+-------+-------+---------------+------------------+---------------------------------+--------------------+--------------+--------------------+-------------------------+--------------+
|2026-01|Central|Industrial     |24871.22          |226328.1                         |4396                |440           |49                  |11.14                    |1             |
|2026-01|Central|Commercial     |10649.03          |87322.05                         |5445                |595           |62                  |10.42                    |2             |
|2026-01|Central|Residential    |814.03            |4636.91                

In [73]:
final_report_cached_df = monthly_energy_reliability_ranked_df.persist(
    StorageLevel.MEMORY_AND_DISK)

final_report_count = final_report_cached_df.count()
print("Final monthly report rows:", final_report_count)

Final monthly report rows: 120


In [76]:
final_report_totals = final_report_cached_df.agg(
    F.round(F.sum("total_billable_kwh"), 3).alias("all_region_site_month_kwh"),
    F.round(F.sum("total_billable_energy_charges_inr"), 2).alias("all_region_site_month_charges_inr"),
    F.sum("total_outage_minutes").alias("all_region_site_month_outage_minutes"))

final_report_totals.show(truncate=False)


+-------------------------+---------------------------------+------------------------------------+
|all_region_site_month_kwh|all_region_site_month_charges_inr|all_region_site_month_outage_minutes|
+-------------------------+---------------------------------+------------------------------------+
|1831426.57               |1.528269644E7                    |791439                              |
+-------------------------+---------------------------------+------------------------------------+



In [75]:
final_report_totals = final_report_cached_df.agg(
    F.round(F.sum("total_billable_kwh"), 3).alias("all_region_site_month_kwh"),
    F.round(F.sum("total_billable_energy_charges_inr"), 2).alias("all_region_site_month_charges_inr"),
    F.sum("total_outage_minutes").alias("all_region_site_month_outage_minutes")
)

final_report_totals.show(truncate=False)

final_report_cached_df.unpersist()
print("Final report cache released.")

capstone_output_path = "/content/energy_monthly_reliability_report_parquet"

(
    monthly_energy_reliability_ranked_df
    .repartition(2)
    .write
    .mode("overwrite")
    .parquet(capstone_output_path)
)

capstone_part_files = [
    f for f in os.listdir(capstone_output_path)
    if f.startswith("part-") and f.endswith(".parquet")
]

print("Capstone Parquet directory:", capstone_output_path)
print("Capstone Parquet part files:", capstone_part_files)
print("Number of Parquet part files:", len(capstone_part_files))

+-------------------------+---------------------------------+------------------------------------+
|all_region_site_month_kwh|all_region_site_month_charges_inr|all_region_site_month_outage_minutes|
+-------------------------+---------------------------------+------------------------------------+
|1831426.57               |1.528269644E7                    |791439                              |
+-------------------------+---------------------------------+------------------------------------+

Final report cache released.
Capstone Parquet directory: /content/energy_monthly_reliability_report_parquet
Capstone Parquet part files: ['part-00001-dce6731d-5dd9-4f5f-83eb-09e9c489ef5b-c000.snappy.parquet', 'part-00000-dce6731d-5dd9-4f5f-83eb-09e9c489ef5b-c000.snappy.parquet']
Number of Parquet part files: 2


In [78]:
import shutil
from google.colab import files

shutil.make_archive(
    "/content/energy_monthly_reliability_report",
    "zip",
    "/content/energy_monthly_reliability_report_parquet")

files.download("/content/energy_monthly_reliability_report.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>